スタッキングモデルを作る。
xgboost+DNNで予測値の推定を行う。


In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from sklearn.model_selection import train_test_split, KFold

今回はデータ加工について、一切行わない。
理由としては、xgboostは、多重共線性を考えなくていい、そして、データ加工があまり必要ではない(最高の精度を目指すならいるが、今回は精度向上を目標にしているわけではない)

In [2]:
california = fetch_california_housing()
df = pd.DataFrame(california.data, columns=california.feature_names)
df['Target'] = california.target
X = df.drop('Target', axis=1).values
y = df['Target'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

今からすることは、ハイパパラメータのチューニングである。
これは、k-foldしている最中に、それぞれのモデルでやるのが理想だが、計算コストの観点から厳しいので、訓練データで最適なハイパパラメータのチューニングをする。

In [3]:
from sklearn.metrics import mean_squared_error
import optuna

In [4]:
def objective(trial):
    param = {
        'random_state': 42,
        'eval_metric': 'rmse',
        # 木の深さ (3〜9の整数)
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        # 学習率 (0.01〜0.3の範囲で対数的に探索)
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        # データの抽出割合 (0.6〜1.0)
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        # 特徴量の抽出割合 (0.6〜1.0)
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        # L2正則化 (1e-3〜10.0の範囲で対数的に探索)
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        
        # アーリーストッピングを効かせるため、木の数は大きめに設定
        'n_estimators': 2000 
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_rmse = []
    for train_idx, valid_idx in kf.split(X_train):
        X_tr, y_tr = X_train[train_idx], y_train[train_idx]
        X_va, y_va = X_train[valid_idx], y_train[valid_idx]
        
        # アーリーストッピングを設定
        model = xgb.XGBRegressor(**param, early_stopping_rounds=50)
        
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False            
        )
        
        preds = model.predict(X_va)
        rmse = np.sqrt(mean_squared_error(y_va, preds))
        cv_rmse.append(rmse)

    return np.mean(cv_rmse)

In [5]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)
print(f"▼ ベストスコア (RMSE): {study.best_value:.4f}")
print("▼ 最も精度の高かったパラメータ:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2026-05-31 14:13:27,867] A new study created in memory with name: no-name-68d9a52a-2b7f-4bab-ae60-88f4fe1b72ae
[I 2026-05-31 14:13:30,117] Trial 0 finished with value: 0.4527245627981733 and parameters: {'max_depth': 6, 'learning_rate': 0.09372382399451941, 'subsample': 0.9522586603141976, 'colsample_bytree': 0.7092009870800877, 'reg_lambda': 0.03380969457950551}. Best is trial 0 with value: 0.4527245627981733.
[I 2026-05-31 14:13:32,647] Trial 1 finished with value: 0.45803002979892904 and parameters: {'max_depth': 9, 'learning_rate': 0.0792535954614789, 'subsample': 0.7852744335758876, 'colsample_bytree': 0.7316166486879622, 'reg_lambda': 0.011893060919202466}. Best is trial 0 with value: 0.4527245627981733.
[I 2026-05-31 14:13:35,324] Trial 2 finished with value: 0.4564243141978547 and parameters: {'max_depth': 6, 'learning_rate': 0.06404523515883782, 'subsample': 0.6377656614741426, 'colsample_bytree': 0.6895728901695638, 'reg_lambda': 0.02036700043805853}. Best is trial 0 with 

▼ ベストスコア (RMSE): 0.4447
▼ 最も精度の高かったパラメータ:
  max_depth: 7
  learning_rate: 0.014578975249579988
  subsample: 0.7753065100840897
  colsample_bytree: 0.736140504686424
  reg_lambda: 0.5405302740919219


比較対象のために、xgboostのチューニングされたハイパパラメータのもとでのxgboost単体でのRMSEを出しておく。


In [6]:
best_params = study.best_params
best_params['random_state'] = 42
best_params['eval_metric'] = 'rmse'
best_params['n_estimators'] = 2000
X_tr, X_va, y_tr, y_va = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)
best_model = xgb.XGBRegressor(**best_params,early_stopping_rounds=50)
best_model.fit(
    X_tr, y_tr,
    eval_set=[(X_va, y_va)], # 切り出した監視用データを渡す
    verbose=100              # 100本ごとにログを表示 (邪魔なら False に)
)

test_preds = best_model.predict(X_test)
final_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
print(f"\n▼ テストデータでのRMSE: {final_rmse:.4f}")


[0]	validation_0-rmse:1.16656
[100]	validation_0-rmse:0.63233
[200]	validation_0-rmse:0.53003
[300]	validation_0-rmse:0.49809
[400]	validation_0-rmse:0.48535
[500]	validation_0-rmse:0.47823
[600]	validation_0-rmse:0.47518
[700]	validation_0-rmse:0.47225
[800]	validation_0-rmse:0.47002
[900]	validation_0-rmse:0.46850
[1000]	validation_0-rmse:0.46677
[1100]	validation_0-rmse:0.46519
[1200]	validation_0-rmse:0.46410
[1300]	validation_0-rmse:0.46329
[1400]	validation_0-rmse:0.46252
[1500]	validation_0-rmse:0.46138
[1600]	validation_0-rmse:0.46071
[1700]	validation_0-rmse:0.45996
[1800]	validation_0-rmse:0.45934
[1900]	validation_0-rmse:0.45899
[1984]	validation_0-rmse:0.45890

▼ テストデータでのRMSE: 0.4421


ここから、このできたハイパパラメータを利用してDNNとのスタッキングモデルを作っていく

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
import xgboost as xgb 
kf = KFold(n_splits=5, shuffle=True, random_state=42)


oof_xgb = np.zeros(len(X_train))
test_xgb = np.zeros(len(X_test))


best_params = study.best_params.copy()
best_params['random_state'] = 42
best_params['eval_metric'] = 'rmse'
best_params['n_estimators'] = 2000

for fold, (train_idx, valid_idx) in enumerate(kf.split(X_train)):
    X_tr, y_tr = X_train[train_idx], y_train[train_idx]
    X_va, y_va = X_train[valid_idx], y_train[valid_idx]

    model = xgb.XGBRegressor(**best_params, early_stopping_rounds=50)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=False
    )

    oof_xgb[valid_idx] = model.predict(X_va) #ここで、OOF予測値を作成
    test_xgb += model.predict(X_test) / kf.n_splits #テストデータの予測値は、各foldの予測値の平均を取る
    
print("OOF予測値の作成完了")

print("\n--- Step 2: DNN用のデータ結合とスケーリング ---")

X_train_combined = np.column_stack((X_train, oof_xgb))#説明変数+OOF予測値
X_test_combined = np.column_stack((X_test, test_xgb))#今はまだ使わない

scaler_dnn_X = StandardScaler()
X_train_dnn_scaled = scaler_dnn_X.fit_transform(X_train_combined)
X_test_dnn_scaled = scaler_dnn_X.transform(X_test_combined)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1)).flatten()

print(f"DNN用 学習データ形状: X={X_train_dnn_scaled.shape}, y={y_train_scaled.shape}")

--- Step 1: XGBoostによるOOF予測の作成 ---
OOF予測値の作成完了

--- Step 2: DNN用のデータ結合とスケーリング ---
DNN用 学習データ形状: X=(16512, 9), y=(16512,)


DNNのモデル設定。本当は、ここのモデルの層や、ハイパパラメータのチューニングをするのがいいが、今回はある程度自分で数値を変えて
そこそこよかったものを選択した。ちなみに、微妙にモデルを変えてもどれもほとんど精度に変化はない

In [69]:
class CascadeDNN(nn.Module):
    def __init__(self, input_dim):
        super(CascadeDNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),  
            nn.ReLU(),    
            
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)       
        )

    def forward(self, x):
        return self.net(x)

X_t = torch.FloatTensor(X_train_dnn_scaled)
y_t = torch.FloatTensor(y_train_scaled).unsqueeze(1) 


batch_size = 64
dataset = TensorDataset(X_t, y_t)#束にしている
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

input_dimension = X_train_dnn_scaled.shape[1]
dnn_model = CascadeDNN(input_dim=input_dimension)
criterion = nn.MSELoss() 
optimizer = optim.Adam(dnn_model.parameters(), lr=0.001, weight_decay=1e-4) # L2正則化を追加

In [70]:
import copy
epochs = 150
patience = 15  
min_delta = 0.00005
best_train_mse = float('inf')
patience_counter = 0
best_model_wts = copy.deepcopy(dnn_model.state_dict())

for epoch in range(epochs):
    dnn_model.train()
    train_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = dnn_model(batch_X)
        loss = criterion(outputs, batch_y) 
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * batch_X.size(0)
    epoch_train_mse = train_loss / len(train_loader.dataset)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:03d}/{epochs} | Train MSE: {epoch_train_mse:.5f}")


    if best_train_mse - epoch_train_mse > min_delta:
        best_train_mse = epoch_train_mse
        best_model_wts = copy.deepcopy(dnn_model.state_dict())
        patience_counter = 0 
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print(f"\n!-- 改善幅が閾値({min_delta})未満の連続 {patience} 回に達しました --!")
        print(f"!-- Epoch {epoch+1} で早期終了します --!")
        break

# 最良の重みをモデルにロード
dnn_model.load_state_dict(best_model_wts)

Epoch 001/150 | Train MSE: 0.28313
Epoch 010/150 | Train MSE: 0.16298
Epoch 020/150 | Train MSE: 0.16056
Epoch 030/150 | Train MSE: 0.15873
Epoch 040/150 | Train MSE: 0.15713
Epoch 050/150 | Train MSE: 0.15814
Epoch 060/150 | Train MSE: 0.15705

!-- 改善幅が閾値(5e-05)未満の連続 15 回に達しました --!
!-- Epoch 61 で早期終了します --!


<All keys matched successfully>

予測精度は、XGboost単体と比べると0.01程下がった。どのハイパパラメータでも、大体このくらいの値になるので
モデル自体は改善できていると考えてもいいかもしれない。

In [71]:
import math
X_test_t = torch.FloatTensor(X_test_dnn_scaled)

dnn_model.eval()
with torch.no_grad():
    test_preds_scaled = dnn_model(X_test_t).numpy() 


final_preds = scaler_y.inverse_transform(test_preds_scaled).flatten()

final_rmse = math.sqrt(mean_squared_error(y_test, final_preds))

print(f" Test RMSE: {final_rmse:.4f}")

 Test RMSE: 0.4326
